# Lab03C — Hipóteses e calibração da inferência

**Objetivo:** avaliar cobertura, falsos positivos e capacidade de detecção do contraste estático Δ = β⁺ − β⁻, sob mecanismos com verdade conhecida. Amostras fixas80/117; família197; três mecanismos ×1.000 réplicas. Não modifica retornos reais e não homologa economicamente os dados.

**Hipótese de pesquisa:** exposições podem diferir entre altas e quedas. A hipótese estatística já utilizada é H₀: Δᵢ=0 contra H₁: Δᵢ≠0, bilateral. Maior beta de queda implica Δ<0; não se muda a alternativa após conhecer resultados. A motivação por risco de queda não é teste de prêmio CAPM. Mudança temporal tem contraste próprio θ; setores são exploratórios. Formulação agora explicitada, **sem pré-registro independente**.

Execute **Ambiente de execução → Executar tudo** em CPU. Não repetir notebooks anteriores. O acesso autenticado ao Drive serve apenas para reutilizar dois ZIPs auditados, obter o pequeno runtime e salvar o novo export. Nenhum download de cotações.

In [ ]:
import os, sys, json, hashlib, zipfile, subprocess, shutil
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd
LOCAL_MODE = bool(os.environ.get('ALEM_BETA_LOCAL'))
os.environ['ALEM_BETA_EXECUTION_ENV'] = 'local preparation' if LOCAL_MODE else 'Colab'
BASE = Path(os.environ.get('ALEM_BETA_LOCAL', '/content/alem_beta_lab03c')).resolve()
BASE.mkdir(parents=True, exist_ok=True)
RUN = BASE / ('run_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
RUN.mkdir()
ROOT = RUN / 'runtime'
try:
    from IPython.display import display, Image
except ImportError:
    def display(value): print(str(value)[:4000])
    def Image(filename): return str(filename)
print('Lab03C |', 'preparação local' if LOCAL_MODE else 'Colab', '| CPU |', RUN.name)

## Entradas e proveniência

Autenticação por SHA256 do ZIP completo e dos arquivos internos. Reutilização dos mesmos80 no intervalo comum; não são novas hipóteses. Preservam-se sete ausências na década (ABCB4/ANIM3/UGPA3) e sete no painel117 recente (UGPA3/AZUL4), sem preenchimento. As quatro AZUL4 são herdadas da matriz; não são novas lacunas produzidas pelo experimento.

In [ ]:
RUNTIME_ID = '1t9sBGr1lMSuGJLoN23CV2Zy4lkMK9Fgu'
RUNTIME_SHA = '3d4ea28835f1499c0bb0c34bcdd91933386642903105fc19f336e23124bcdca6'
X_ID = '1aExT7v-GD8s5xKzM0K-0LIIJER8E8Twg'
B_ID = '1hhLvxQpAnEGopDvnt1Mh9IbkV0gYt9Gb'
X_SHA = '297da2afde534082c398324f308df7af134c9692e7d61d6e0b2b820341ea5214'
B_SHA = '645946c0bfee05cc7cbbc34b63af74700fcc7e0a3d3ac14ee7aaa398323fa26d'
FOLDER_ID = '1Wb3lD5Fiz0orMkUqu0g-jiQFEGQQFDll'
def digest(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
if not LOCAL_MODE:
    from google.colab import auth, drive
    auth.authenticate_user()
    drive.mount('/content/drive')
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
    credentials, _ = google.auth.default()
    service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    CACHE = Path('/content/drive/MyDrive/alem_do_beta/lab03c_cache')
    CACHE.mkdir(parents=True, exist_ok=True)
else:
    CACHE = BASE / 'cache'; CACHE.mkdir(exist_ok=True)
    LOCAL_FILES = {RUNTIME_ID:BASE/'lab03c_runtime_v1.zip', X_ID:BASE.parent/'work_lab03b_colab/x.zip', B_ID:BASE.parent/'work_lab03b_colab/results.zip'}
def obtain(file_id, expected):
    if LOCAL_MODE:
        p = LOCAL_FILES[file_id]; assert digest(p)==expected; return p
    cached = CACHE/(file_id+'.zip')
    if cached.exists() and digest(cached)==expected:
        print('Cache autenticado:',file_id); return cached
    name = service.files().get(fileId=file_id,fields='name').execute()['name']
    # Existing exports anywhere in MyDrive are tried by exact name and hash.
    for candidate in Path('/content/drive/MyDrive').rglob(name):
        if candidate.is_file() and digest(candidate)==expected:
            print('Export existente autenticado:',name); return candidate
    stage = cached.with_suffix('.part')
    with stage.open('wb') as stream:
        request = service.files().get_media(fileId=file_id)
        download = MediaIoBaseDownload(stream,request); done=False
        while not done: _,done=download.next_chunk()
    assert digest(stage)==expected, 'SHA256 divergente; não utilizar arquivo'
    stage.replace(cached); print('Artefato de projeto autenticado:',name); return cached
RUNTIME = obtain(RUNTIME_ID,RUNTIME_SHA)
X_ARCHIVE = obtain(X_ID,X_SHA)
B_ARCHIVE = obtain(B_ID,B_SHA)

In [ ]:
ROOT.mkdir()
with zipfile.ZipFile(RUNTIME) as z:
    names=z.namelist(); assert len(names)==len(set(names))
    for n in names: assert not n.startswith('/') and '..' not in Path(n).parts
    z.extractall(ROOT)
source_manifest=json.loads((ROOT/'data/lab03c/runtime_manifest.json').read_text())
for name, expected in source_manifest.items(): assert digest(ROOT/name)==expected, name
assert set(names)==set(source_manifest)|{'data/lab03c/runtime_manifest.json'}
lock=json.loads((ROOT/'data/lab03c/inputs_manifest.json').read_text())
assert digest(ROOT/'data/lab03c/plano_v1.json')==lock['plan_sha256']
plan=json.loads((ROOT/'data/lab03c/plano_v1.json').read_text())
assert plan['replications']==1000 and plan['primary_cases']==197
sys.path.insert(0,str(ROOT/'src'))
print(len(source_manifest),'fontes autenticadas; protocolo congelado antes dos resultados de simulação.')

## Modelo gerador e critérios

Mercado observado e calendários são fixos. Geram-se117 caminhos de erro por réplica, reutilizados nas janelas sobrepostas. Escala residual e beta base são calibrados uma vez por ativo. O modelo é `r=β_base·m+δ·m·1(m>0)+σ·h·u`. Inovações têm componente comum0,3, escolhido como cenário, sem afirmar covariância empírica estimada.

Mecanismos: normal independente; t5 padronizada com escala por mercado defasado; mesma t5 e escala com AR(0,3). Burn-in256, semente20261023 por mecanismo/réplica. Sete efeitos densos δ∈{−0,50;−0,25;−0,10;0;0,10;0,25;0,50}; dois esparsos com ±0,25 nos primeiros12 ativos únicos ordenados por identificador, totalizando22 casos por sobreposição. Não se escolhem ativos por significância.

HAC5 é primário; HAC0/20 são sensibilidades. Mantêm-se interceptos separados, 200 pares/60 por regime/95% cobertura, calendário original, Bartlett, correção n/(n−4), aproximação normal e Holm197. A linearidade OLS permite injetar δ sem repetir ajustes idênticos: muda o coeficiente, não os resíduos ou a covariância. Essa identidade será conferida por regressões completas independentes.

In [ ]:
command=[sys.executable,'-m','unittest','discover','-s',str(ROOT/'tests'),'-p','test_lab03c.py','-v']
check=subprocess.run(command,capture_output=True,text=True)
print(check.stdout+check.stderr)
assert check.returncode==0, 'Testes falharam: interromper'

In [ ]:
from lab03c import prepare
prepared=prepare(X_ARCHIVE,B_ARCHIVE,lock)
display(prepared['design'].groupby('scenario').agg(ativos=('papel_id','size'),sessoes=('n_sessions','first'),ausencias=('n_missing','sum')))
assert prepared['design'].groupby('scenario').n_missing.sum().to_dict()=={'decada':7,'recente':7}
display(prepared['nuisance'].head())

## Execução do painel inteiro

1.000 réplicas em cada mecanismo, sem lotes de cinco ativos, sem descartar/redesenhar réplicas desfavoráveis. O progresso aparece a cada100. A duração depende da CPU; na preparação local o núcleo levou cerca de2min. Aguarde também gráficos e conferência independente. Não interpretar valores simulados como novos retornos observados.

In [ ]:
OUTPUT=RUN/'outputs'
subprocess.run([sys.executable,'-u',str(ROOT/'scripts/run_lab03c.py'),'--x-archive',str(X_ARCHIVE),'--b-archive',str(B_ARCHIVE),'--output',str(OUTPUT)],check=True)
print((OUTPUT/'metricas.json').read_text())

In [ ]:
subprocess.run([sys.executable,str(ROOT/'scripts/plot_lab03c.py'),'--output',str(OUTPUT)],check=True)
for name in ['lab03c_calibracao','lab03c_rejeicao','lab03c_esparso']:
    display(Image(filename=str(OUTPUT/'figures'/(name+'.png'))))

## Conferência independente

Sem importar a implementação nova: reconstrói inovações, calibração e máscaras; ajusta1773 regressões completas com interceptos próprios e efeito injetado; reconfere162mil valores das quatro tabelas a partir das réplicas arquivadas. Conferência aritmética não homologa os dados financeiros nem garante cobertura nominal.

In [ ]:
subprocess.run([sys.executable,'-u',str(ROOT/'scripts/validate_lab03c_outputs.py'),'--output',str(OUTPUT),'--x-archive',str(X_ARCHIVE)],check=True)
receipt=json.loads((OUTPUT/'validacao_independente.json').read_text())
assert receipt['status']=='passed' and receipt['full_design_oracle_fits']==1773
print(json.dumps(receipt,indent=2))

In [ ]:
panel=pd.read_csv(OUTPUT/'panel_summary.csv'); family=pd.read_csv(OUTPUT/'family.csv')
display(panel[panel.method.eq('HAC5') & panel.pattern.eq('dense_+0.00')][['mechanism','scenario','coverage','coverage_mcse','type1_raw']])
display(family[family.method.eq('HAC5') & family.pattern.eq('dense_+0.00')][['mechanism','fwer_holm','fwer_holm_mc_low','fwer_holm_mc_high']])
display(panel[panel.method.eq('HAC5') & panel.pattern.eq('dense_+0.25')][['mechanism','scenario','power_raw','power_holm','power_holm_mcse']])

## Interpretação e exportação

Cobertura alvo95%; FWER alvo5%. Desvios devem ser reportados, sem substituir HAC5 pela configuração que rejeita mais. Barras de cobertura/poder são incerteza Monte Carlo entre réplicas, preservando dependência entre ativos; FWER usa Wilson95%, inclusive zero sucessos. R=1.000 implica erro padrão binomial máximo1,581 ponto percentual. δ=0 mede erroI; nos demais efeitos mede capacidade de detecção. Poder depende do cenário e **não é poder observado retrospectivo**, margem de equivalência ou limiar econômico.

A preparação local encontrou cobertura inferior ao nominal em cenários e inflação da FWER em alguns deles. Consulte os valores desta execução. Isso limita a força inferencial das rejeições empíricas; não prova simetria. O experimento não calibra a família temporal474, não conclui auditoria global0/197, não demonstra persistência futura nem prêmio CAPM.

O ZIP completo contém réplicasNPZ, tabelas por ativo/regime, três figuras PNG/PDF, plano, fontes autenticadas e versões do ambiente. **Salve também este notebook no Colab após executar tudo**, para conferência da execução efetiva.

Referências: Ang, Chen e Xing (2006), DOI10.1093/rfs/hhj035; Newey e West (1987), DOI10.2307/1913610; Holm (1979), DOI10.2307/4615733; Morris, White e Crowther (2019), DOI10.1002/sim.8086. O último orienta planejamento e erro Monte Carlo; os mecanismos financeiros são escolhas explícitas deste estudo.

In [ ]:
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
EXPORT=RUN/('resultados_lab03c_'+STAMP+'.zip')
subprocess.run([sys.executable,str(ROOT/'scripts/export_lab03c.py'),'--output',str(OUTPUT),'--destination',str(EXPORT)],check=True)
if not LOCAL_MODE:
    saved=service.files().create(body={'name':EXPORT.name,'parents':[FOLDER_ID]},media_body=MediaFileUpload(str(EXPORT),mimetype='application/zip',resumable=True),fields='id,name,webViewLink').execute()
    print('Resultado salvo no Drive:',saved.get('webViewLink','https://drive.google.com/file/d/'+saved['id']+'/view'))
else:
    print('Preparação local; execução Colab ainda pendente.')
print('SHA256 do ZIP:',digest(EXPORT))
print('Concluído. Salve o notebook no Colab.')